**Insights:**

--Every emotionally charged sentence (joy,sadness,anger etc) can be sarcastic or not sarcastic(Purely happiness,sadness,anger etc)

--ML wont be able to detect sarcasam unless whole sentence match with the sentence/phrase in sarcasam dict 

--I would first lable sentence as joy,sadness,anger ---> then will use weak sarcasam dict -->to lable --->manually annotate as much as possible -->as for neutral!=sarcasam we know that

**--no neutral also have to check**

Testing :

Approach 1) when a sentence comes label through emotion
Approach 2 )Based on approach 2 ask model to see if it has touch of sarcasam

Understanding Sarcasam

##  Definition 

**Sarcasm** is when someone says something **positive or neutral on the surface**,
but the **real meaning (intent)** is **negative, mocking, or opposite** of the words used.

It’s a form of **verbal irony** used to **express ridicule, criticism, or humor** — often through **tone, exaggeration, or contrast**.



## **Key Characteristics of Sarcasm**

| Feature                  | Description                                                                      |
| ------------------------ | -------------------------------------------------------------------------------- |
| **Contradiction**        | The literal meaning is opposite to the intended emotion.                         |
| **Tone or exaggeration** | Overly positive words or emojis (“wah”, “amazing 😂”) for bad situations.        |
| **Context dependency**   | Needs situation to understand — “Great!” can be sincere or sarcastic.            |
| **Common markers**       | 😂 😏 🙄 🤦‍♀️ “wah bhai wah”, “great job”, “kya baat hai”, “wah kya timing hai” |

---

##  **Why It’s Hard for Machines**

Sarcasm is **not word-based** — it’s **context-based**:

* Literal meaning ≠ intended meaning.
* Same sentence can be sarcastic or sincere depending on **context**, **tone**, or **emoji**.



In [75]:
import pandas as pd

# Load the CSV
emotion_df = pd.read_csv('/kaggle/input/emotion-key-words/Emotion_dictt.csv')

csv_file_path = '/kaggle/input/emotion-variation-file/EV.csv'  # replace with your CSV
dictionary = load_emotion_dictionaries(csv_file_path)

# Build sarcasm dictionary from CSV
sarcasm_df = emotion_df[emotion_df['emotion_category'].str.lower() == 'sarcasm']
sarcasm_words = sarcasm_df['Word'].str.lower().tolist()

print("Sarcasm words:")
print(sarcasm_words)

# Keep only actual emotions, exclude sarcasm
emotion_df = emotion_df[~emotion_df['emotion_category'].str.lower().isin(['sarcasm'])]

# Create a dictionary: word -> emotion
word_to_emotion = dict(zip(emotion_df['Word'].str.lower(), emotion_df['emotion_category'].str.lower()))

print("\nWord to emotion mapping:")
for word, emotion in list(word_to_emotion.items())[:20]:  # print first 20 for readability
    print(f"{word} -> {emotion}")


Sarcasm words:
['lol', 'wow', 'mid', 'cringe', 'skibidi', 'sigma', 'rizz', 'gyatt', 'grimace', 'bussin', 'no cap', 'meh', 'bruh', 'oof', 'yikes', 'oohs', 'claps', 'encore', 'standing ovation', 'slow clap', 'lmao', 'lmfao', 'rofl', 'xd', 'xdxd', 'lul', 'kek', 'pfft', 'periodt', 'whatever']

Word to emotion mapping:
mubarak -> joy
khushi -> joy
khush -> joy
maza -> joy
khob -> joy
zabardast -> joy
shaandaar -> joy
mast -> joy
josh -> joy
hasna -> joy
masti -> joy
wahwah -> joy
khushion -> joy
khushhaal -> joy
dilchaspi -> joy
muskurahat -> joy
achi -> joy
khoshgwar -> joy
khilkhilana -> joy
muhabbat -> joy


In [76]:
texts = [
    "aaj maza aya lol",
    "wah bhai wah, net gaya phir 😂",
    "kya baat hai, exam fail hogaya!",
    "bohat mazay ka kaam kiya lol"
]

# Normalize
texts_normalized = [normalize_text(text, dictionary) for text in texts]

# Build DataFrame
df = pd.DataFrame(texts_normalized, columns=['text'])

# Apply emotion labeling
dominant_emotions = []
words_per_emotion = []
for text in df['text']:
    dom_emotion, words_str = label_emotion_compact(text)
    dominant_emotions.append(dom_emotion)
    words_per_emotion.append(words_str)

df['emotion_detected'] = dominant_emotions
df['words_of_each_emotion'] = words_per_emotion

# Apply sarcasm labeling
sarcasm_found = []
sarcasm_labels = []
for text in df['text']:
    phrases, label = label_sarcasm(text, sarcasm_words)
    sarcasm_found.append(phrases)
    sarcasm_labels.append(label)

df['sarcasm_words'] = sarcasm_found
df['sarcasm'] = sarcasm_labels

# -----------------------------
# Print final compact table
# -----------------------------
print(df.to_string(index=False))


                         text emotion_detected   words_of_each_emotion sarcasm_words  sarcasm
             aaj maza aya lol              joy maza(joy), aaj(neutral)           lol        1
 wah bhai wah net gaya phir 😂          neutral           phir(neutral)                      0
kya baat hai exam fail hogaya          neutral           baat(neutral)                      0
  bohat mazay ka kam kiya lol          neutral            kam(neutral)           lol        1


maza--->mazay  is not mapped rightly so it could not detect happy emotion
emotion word variations not right
acha --> is not included in joy dict
fail---> not in sadness

**overall emotion words and spelling dict is not made rightly**


--here neutral has sarcastic tone so also see neutral


In [ ]:



##Create Sarcasam rules to teach to tranformers 
#not in coding ,created for help
# [
#   {
#     "phrase": "wah",
#     "category": "lexical_cue",
#     "rule": "if near [complaint|negative word|fail|problem], label sarcasm"
#   },
#   {
#     "phrase": "bohat acha",
#     "category": "lexical_cue",
#     "rule": "if near negative context or emoji 😒 😂, label sarcasm"
#   },
#   {
#     "phrase": "great job",
#     "category": "lexical_cue",
#     "rule": "if near complaint/negative, label sarcasm"
#   },
#   {
#     "phrase": "😂",
#     "category": "emoji_cue",
#     "rule": "sarcasm if near complaint or contradiction"
#   },
#   {
#     "phrase": "🙄",
#     "category": "emoji_cue",
#     "rule": "sarcasm if near positive words + negative context"
#   }
# ]  
# ## Create Sarcasam weak dictionary for labeling sentences







sarcasm_dict = [
    # Lexical cues / phrases
    {"phrase": "wah", "category": "lexical_cue", "rule": "if near negative context, label sarcasm"},
    {"phrase": "bohat acha", "category": "lexical_cue", "rule": "if near complaint/failure, label sarcasm"},
    {"phrase": "great job", "category": "lexical_cue", "rule": "if near failure/problem, label sarcasm"},
    {"phrase": "kya baat hai", "category": "lexical_cue", "rule": "if near complaint/problem, label sarcasm"},
    {"phrase": "kya scene hai", "category": "lexical_cue", "rule": "if near negative context, label sarcasm"},
    
    # Single words commonly used sarcastically
    {"phrase": "lol", "category": "lexical_cue", "rule": "sarcasm if used in complaint/failure context"},
    {"phrase": "haha", "category": "lexical_cue", "rule": "sarcasm if used in complaint/failure context"},
    {"phrase": "wow", "category": "lexical_cue", "rule": "sarcasm if near negative context"},
    {"phrase": "nice", "category": "lexical_cue", "rule": "sarcasm if near negative context"},
    {"phrase": "good", "category": "lexical_cue", "rule": "sarcasm if near negative context"},
    
    # Emojis commonly indicating sarcasm
    {"phrase": "😂", "category": "emoji_cue", "rule": "sarcasm if near complaint or contradiction"},
    {"phrase": "🙄", "category": "emoji_cue", "rule": "sarcasm if near positive words + negative context"},
    {"phrase": "😒", "category": "emoji_cue", "rule": "sarcasm if near complaint or failure"},
    
    # Social media slang / modern cues
    {"phrase": "pog", "category": "lexical_cue", "rule": "sarcasm if exaggerated context"},
    {"phrase": "based", "category": "lexical_cue", "rule": "sarcasm if exaggerated context"},
    {"phrase": "ratio", "category": "lexical_cue", "rule": "sarcasm if used ironically"},
    {"phrase": "mid", "category": "lexical_cue", "rule": "sarcasm if used ironically"},
    {"phrase": "sus", "category": "lexical_cue", "rule": "sarcasm if used ironically"}
]


#make sepaarte file for sarcasam dict 
#make separte for all emotiosn so es to modify
#weak labelling 

#scrap after sarcasam rightly labelled


In [ ]:
#Scrap data from reddit 


#do spelling correction 



#lable data 


# Manually lable 


#download final training data